# Lotto Modeling Suite

This notebook is responsible for model construction and experiment generation.

Models included:
1. Frequency heuristic baseline
2. Gap heuristic baseline
3. Random baseline
4. Logistic regression
5. Random forest
6. XGBoost
7. Classifier chain

The notebook trains or generates all model outputs, runs the rolling backtest, and saves reusable evaluation artifacts for the next notebook.

### 1. Library Imports

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
APP_ROOT = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "src" / "notebook_support.py").exists() and (candidate / "data").exists():
        APP_ROOT = candidate
        break
    if (candidate / "app" / "src" / "notebook_support.py").exists() and (candidate / "app" / "data").exists():
        APP_ROOT = candidate / "app"
        break

if APP_ROOT is None:
    raise RuntimeError(f"Could not locate app root from {cwd}")

if str(APP_ROOT) not in sys.path:
    sys.path.insert(0, str(APP_ROOT))

from src.notebook_support import describe_notebook_environment
describe_notebook_environment(APP_ROOT)
import pandas as pd

from src.config import MODEL_OUTPUT_DIR
from src.pipelines import run_model_pipeline
from src.visualization import save_report_table


### 2. Experiment Configuration

Adjust these values if you want to change the feature window, official feature-set selection, holdout split size, execution mode, or rolling backtest setup.

Use `RUN_MODE = "quick"` for a fast iteration loop.
Use `RUN_MODE = "full"` for the full baseline suite.

Supported `FEATURE_SET_NAME` values come from the shared feature registry:

- `base`
- `base_plus_pattern`
- `base_plus_context`
- `full_feature_set`


In [2]:
RUN_MODE = "full"

WINDOW = 20
FEATURE_SET_NAME = "base"
TEST_RATIO = 0.2
RANDOM_SEED = 42
TRAIN_WINDOW_SIZE = None  # set to 200/300/500 for recent-window training

BACKTEST_INITIAL_TRAIN_SIZE = 600
BACKTEST_TEST_SIZE = 30
BACKTEST_STEP_SIZE = 30

# Quick mode keeps the notebook responsive while you validate the pipeline.
if RUN_MODE == "quick":
    HOLDOUT_MODEL_NAMES = ["freq_heuristic", "gap_heuristic", "random_baseline", "logistic_regression"]
    BACKTEST_MODEL_NAMES = ["random_baseline", "logistic_regression"]
    INCLUDE_BACKTEST = True
    MAX_BACKTEST_FOLDS = 3
else:
    HOLDOUT_MODEL_NAMES = None
    BACKTEST_MODEL_NAMES = None
    INCLUDE_BACKTEST = True
    MAX_BACKTEST_FOLDS = None


### 3. Run the Modeling Suite

This cell prepares the aligned dataset, runs the selected holdout models, optionally computes the rolling backtest, and saves the outputs under the model output directory.

In [3]:
results = run_model_pipeline(
    window=WINDOW,
    test_ratio=TEST_RATIO,
    random_seed=RANDOM_SEED,
    backtest_initial_train_size=BACKTEST_INITIAL_TRAIN_SIZE,
    backtest_test_size=BACKTEST_TEST_SIZE,
    backtest_step_size=BACKTEST_STEP_SIZE,
    holdout_model_names=HOLDOUT_MODEL_NAMES,
    backtest_model_names=BACKTEST_MODEL_NAMES,
    include_backtest=INCLUDE_BACKTEST,
    max_backtest_folds=MAX_BACKTEST_FOLDS,
    feature_set_name=FEATURE_SET_NAME,
    train_window_size=TRAIN_WINDOW_SIZE,
)
results["metadata"]


{'window': 20,
 'test_ratio': 0.2,
 'random_seed': 42,
 'feature_set_name': 'base',
 'available_feature_sets': ['base',
  'base_plus_pattern',
  'base_plus_context',
  'full_feature_set'],
 'holdout_model_names': ['freq_heuristic',
  'gap_heuristic',
  'random_baseline',
  'logistic_regression',
  'random_forest',
  'xgboost',
  'classifier_chain',
  'soft_voting_ensemble'],
 'backtest_model_names': ['freq_heuristic',
  'gap_heuristic',
  'random_baseline',
  'logistic_regression',
  'soft_voting_ensemble'],
 'include_backtest': True,
 'backtest_initial_train_size': 600,
 'backtest_test_size': 30,
 'backtest_step_size': 30,
 'max_backtest_folds': None,
 'n_rows': 1202,
 'n_features': 90,
 'n_train_rows': 961,
 'n_test_rows': 241,
 'train_window_size': None}

### 4. Saved Holdout Summary

In [4]:
results["holdout_summary"]

,model,subset_accuracy,number_level_accuracy,avg_hit,hit_std,precision_at_6,recall_at_6,brier_score
7,soft_voting_ensemble,0.0,0.775934,0.958506,0.838852,0.159751,0.159751,0.193813
6,classifier_chain,0.0,0.774643,0.929461,0.804038,0.154910,0.154910,0.199375
3,logistic_regression,0.0,0.772430,0.879668,0.843600,0.146611,0.146611,0.305388
5,xgboost,0.0,0.772245,0.875519,0.879139,0.145920,0.145920,0.175720
4,random_forest,0.0,0.768373,0.788382,0.751782,0.131397,0.131397,0.137277
2,random_baseline,0.0,0.768188,0.784232,0.764297,0.130705,0.130705,NaN
0,freq_heuristic,0.0,0.766897,0.755187,0.763688,0.125864,0.125864,NaN
1,gap_heuristic,0.0,0.765606,0.726141,0.751003,0.121024,0.121024,NaN


### 5. Saved Rolling Backtest Summary

In [5]:
results["backtest_summary"]

,model,folds,mean_subset_accuracy,mean_number_level_accuracy,mean_avg_hit,mean_precision_at_6,mean_recall_at_6,mean_brier_score,std_avg_hit
4,soft_voting_ensemble,20,0.0,0.771630,0.861667,0.143611,0.143611,0.194219,0.141948
3,random_baseline,20,0.0,0.771556,0.860000,0.143333,0.143333,NaN,0.155447
2,logistic_regression,20,0.0,0.770667,0.840000,0.140000,0.140000,0.307935,0.140425
1,gap_heuristic,20,0.0,0.769259,0.808333,0.134722,0.134722,NaN,0.177664
0,freq_heuristic,20,0.0,0.767852,0.776667,0.129444,0.129444,NaN,0.139799


### 6. Output Files

The next notebook reads these saved artifacts instead of retraining all models again.

In [6]:
pd.Series({key: str(value) for key, value in results["paths"].items()})

holdout_summary                      /workspace/models/results/holdout_summary.csv
holdout_draw_results             /workspace/models/results/holdout_draw_results...
backtest_results                    /workspace/models/results/backtest_results.csv
backtest_summary                    /workspace/models/results/backtest_summary.csv
metadata                               /workspace/models/results/run_metadata.json
artifact_logistic_regression     /workspace/models/artifacts/logistic_regressio...
artifact_random_forest            /workspace/models/artifacts/random_forest.joblib
artifact_xgboost                        /workspace/models/artifacts/xgboost.joblib
artifact_classifier_chain        /workspace/models/artifacts/classifier_chain.j...
artifact_soft_voting_ensemble    /workspace/models/artifacts/soft_voting_ensemb...
dtype: object

### 7. Summary

This notebook now focuses on modeling only.

It prepares the aligned dataset, runs the selected baselines and machine-learning models, optionally executes the rolling backtest, and saves the holdout and backtest outputs as reusable files.

The next notebook is dedicated to evaluation, comparison, visualization, and interpretation of these saved results.

### Report Export


In [7]:
# Save key model outputs for the final report
save_report_table(results["holdout_summary"], "table_05_holdout_summary.csv")
save_report_table(results["backtest_summary"], "table_06_backtest_summary.csv")
save_report_table(pd.DataFrame([results["metadata"]]), "table_08_run_metadata.csv")
save_report_table(pd.Series({key: str(value) for key, value in results["paths"].items()}).rename_axis("artifact").reset_index(name="path"), "table_09_model_output_paths.csv")
print("Saved model baseline report artifacts.")


Saved model baseline report artifacts.
